# Multi-agent systems and delegation

**Module:** Module 2 — Advanced Agent Patterns  
**Notebook:** `notebooks/module-2/2.3_multi_agent.ipynb`

## Purpose
Create specialist agents and expose them as tools so a parent agent can delegate bounded tasks.

## Workflow
1. **Prepare the environment** — load required packages and local environment variables. Keep API keys in an ignored `.env` file; never paste credentials into notebook cells.
2. **Build the capability** — configure the model/agent, messages, tools, middleware or data source used in this lesson.
3. **Execute the example** — invoke or stream the agent and inspect the returned messages, tool calls, structured response or state.
4. **Validate behavior** — compare the result with the intended behavior, test an edge case, and check provider/service availability if execution fails.
5. **Reflect** — use the key learnings below to explain the design choices and limitations.

## Learning objectives
- Understand the concept demonstrated by this notebook and where it fits in an agent workflow.
- Explain the role of the main LangChain/LangGraph abstractions used in the implementation.
- Trace inputs through execution and interpret the resulting output.
- Identify operational concerns such as credentials, external services, state, safety and error handling.

## Key learnings
- MCP standardizes how an application exposes tools and resources to clients.
- Context is for runtime dependencies; state is for data that belongs to the agent execution/thread.
- Multi-agent design works best when each specialist has a narrow responsibility and clear input/output.

## Prerequisites and execution notes
- Use the repository setup instructions in the root `README.md` and the supported Python version.
- Configure only the provider credentials needed by this notebook in your local `.env` file.
- Some cells call external APIs, local MCP services, audio devices or sample databases; those dependencies must be available.
- Model output is non-deterministic and provider model names/API behavior can change. Treat outputs as examples, not fixed expected strings.

---



### Implementation step 1

Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from dotenv import load_dotenv. Loads local environment variables so provider/API credentials can be read without hard-coding secrets in the notebook.

from dotenv import load_dotenv

load_dotenv()

## Creating subagents

### Implementation step 3

Imports the libraries and APIs needed by the following example: from langchain.tools import tool. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.tools import tool. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent.

from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

### Implementation step 4

Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.agents import create_agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example.

from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

### Implementation step 6

Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from langchain.messages import HumanMessage. Defines a reusable Python function/tool. The docstring and type annotations communicate the tool purpose and input schema to the agent. Configures a chat model or agent. Model choice, system instructions and attached tools determine the capabilities available in this example. Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response. Inspects or prints the result so you can verify the output structure and follow the execution path.

from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

### Implementation step 8

Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Executes the model/agent with an input payload. Inspect the returned object to understand messages, structured data, tool activity or state changes. Builds a typed conversation message. Message roles and ordering provide the context the model uses for its next response.

question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

### Implementation step 9

Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

**What to verify:** Check that the output matches the intended data flow, and inspect intermediate messages/tool calls when behavior is unclear.


In [ ]:
# Purpose: Imports the libraries and APIs needed by the following example: from pprint import pprint. Inspects or prints the result so you can verify the output structure and follow the execution path.

from pprint import pprint

pprint(response)